In [2]:
import json
from pathlib import Path

import pandas as pd
import yaml
import numpy as np
import structlog

from common import read_day, daterange

with open("../config.copenhagen_sample.yaml", "r") as fp:
    config = yaml.safe_load(fp)

days = []
for i in config["days"]:
    days += list(daterange(i["start"], i["end"]))

log = structlog.get_logger()

In [3]:
OUTPUT = f"{config['dirs']['output']}/{config['city']}/id_lookup"

In [4]:
ids = []
for i in days:
    temp = read_day(i, config["dirs"]["raw_source"])
    dids = temp.unique(subset=[config["id_column"]], keep="first")[
        config["id_column"]
    ].to_list()
    ids += dids
# runs 25 minutes for Budapest data

In [5]:
log.info(f"number unique devices: {len(np.unique(ids))}")

2026-04-22 11:31:48 [info     ] number unique devices: 96725  


In [6]:
lookup = {}
for i, device_id in enumerate(np.unique(ids)):
    lookup[device_id] = i + 1

In [7]:
Path(OUTPUT).mkdir(parents=True, exist_ok=True)
with open(f"{OUTPUT}/id_lookup.json", "w") as fp:
    json.dump(lookup, fp, indent=1)

## create partitions from original IDs

In [8]:
with open(f"{OUTPUT}/id_lookup.json", "r") as fp:
    lookup = json.load(fp)

In [9]:
partition = {v: k[:2] for k, v in lookup.items()}

In [10]:
with open(f"{OUTPUT}/partition.json", "w") as fp:
    json.dump(partition, fp, indent=1)

### descriptives

In [11]:
partition_distribution = (
    pd.DataFrame({"partition": partition.values(), "device_id": partition.keys()})
    .groupby("partition")["device_id"]
    .count()
    .reset_index()
)
partition_distribution["device_id"].describe()

count    256.000000
mean     377.832031
std       19.991349
min      309.000000
25%      364.750000
50%      376.500000
75%      392.000000
max      424.000000
Name: device_id, dtype: float64